# LATINO Super-Resolution Experiment

In [ ]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Add project root to path
import sys
sys.path.insert(0, '.')

from models.lcm import LCMWrapper, get_device
from operators.downsample import DownsampleOperator
from solver.latino import LATINOSolver
from metrics.psnr import psnr
from metrics.ssim import ssim

## Configuration

In [ ]:
# Option 1: CelebA (faces) - downloads ~1.4GB first time
from torchvision.datasets import CelebA
from torchvision import transforms

transform = transforms.Compose([
    transforms.CenterCrop(178),  # Crop to square
    transforms.Resize(256),
    transforms.ToTensor(),
])

dataset = CelebA(root="./data", split="test", download=True, transform=transform)
gt_image = dataset[0][0].unsqueeze(0)  # [1, 3, 256, 256]
print(f"CelebA image shape: {gt_image.shape}")

# # Option 2: Flowers102 - smaller download (~350MB)
# from torchvision.datasets import Flowers102
# transform = transforms.Compose([
#     transforms.Resize(256),
#     transforms.CenterCrop(256),
#     transforms.ToTensor(),
# ])
# dataset = Flowers102(root="./data", split="test", download=True, transform=transform)
# gt_image = dataset[0][0].unsqueeze(0)

# # Option 3: STL10 (96x96 upscaled to 256) - ~2.5GB
# from torchvision.datasets import STL10
# transform = transforms.Compose([
#     transforms.Resize(256),
#     transforms.ToTensor(),
# ])
# dataset = STL10(root="./data", split="test", download=True, transform=transform)
# gt_image = dataset[0][0].unsqueeze(0)

# # Option 4: Download a sample image from URL
# import urllib.request
# from PIL import Image
# url = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"
# urllib.request.urlretrieve(url, "lenna.png")
# img = Image.open("lenna.png").convert("RGB").resize((256, 256))
# gt_image = transforms.ToTensor()(img).unsqueeze(0)

# Settings
SCALE_FACTOR = 4
NUM_ITERATIONS = 4
DELTA = 1.0
TIMESTEPS = [800, 600, 400, 200]
NOISE_LEVEL = 0.0

# Auto-detect device
DEVICE = get_device("auto")
DTYPE = torch.float32 if DEVICE != "cuda" else torch.float16

print(f"Device: {DEVICE}")
print(f"Dtype: {DTYPE}")
print(f"Image shape: {gt_image.shape}")

## Load and Prepare Image

In [ ]:
def show_images(images, titles, figsize=(15, 5)):
    """Display multiple images side by side"""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if img.dim() == 4:
            img = img.squeeze(0)
        img_np = img.permute(1, 2, 0).cpu().numpy()
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Image already loaded from dataset above
print(f"Image shape: {gt_image.shape}")

# Move to device
gt_image = gt_image.to(DEVICE)

# Show the image
show_images([gt_image], ["Ground Truth (256x256)"])

## Create Degraded Observation

In [ ]:
# Create downsampling operator
operator = DownsampleOperator(scale_factor=SCALE_FACTOR)

# Test adjoint property
x_shape = gt_image.shape
y_shape = operator.get_output_shape(x_shape)
passed, rel_error = operator.test_adjoint(x_shape, y_shape, device=DEVICE)
print(f"Adjoint test: {'PASSED' if passed else 'FAILED'} (rel_error={rel_error:.2e})")

# Create degraded observation
y = operator.forward(gt_image)

if NOISE_LEVEL > 0:
    y = y + torch.randn_like(y) * NOISE_LEVEL
    y = torch.clamp(y, 0.0, 1.0)
    print(f"Added noise with sigma={NOISE_LEVEL}")

print(f"Observation shape: {y.shape}")

# Bicubic baseline (on CPU for MPS compatibility)
bicubic = torch.nn.functional.interpolate(
    y.cpu(), scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False
).to(DEVICE)
bicubic = torch.clamp(bicubic, 0.0, 1.0)

bicubic_psnr = psnr(bicubic, gt_image)
bicubic_ssim = ssim(bicubic, gt_image)
print(f"Bicubic baseline: PSNR={bicubic_psnr:.2f} dB, SSIM={bicubic_ssim:.4f}")

show_images([y, bicubic], [f"Low-res ({y.shape[-2]}x{y.shape[-1]})", f"Bicubic (PSNR={bicubic_psnr:.2f})"])

## Load LCM Model

In [ ]:
# Load LCM (this may take a while on first run)
lcm = LCMWrapper(device=DEVICE, dtype=DTYPE)
print("LCM loaded successfully!")

## Run LATINO Solver

In [ ]:
# Create solver
solver = LATINOSolver(
    lcm_model=lcm,
    operator=operator,
    delta=DELTA,
    timesteps=TIMESTEPS,
    device=DEVICE,
)

# Track intermediate results
intermediates = []

def callback(k, x):
    intermediates.append(x.cpu().clone())
    iter_psnr = psnr(x, gt_image)
    iter_ssim = ssim(x, gt_image)
    print(f"  Iteration {k+1}: PSNR={iter_psnr:.2f} dB, SSIM={iter_ssim:.4f}")

# Run LATINO
print("Running LATINO solver...")
result = solver.solve(
    y=y,
    num_iterations=NUM_ITERATIONS,
    callback=callback,
    verbose=True,
)

# Final metrics
final_psnr = psnr(result, gt_image)
final_ssim = ssim(result, gt_image)

print("\n" + "="*50)
print("RESULTS")
print("="*50)
print(f"Bicubic:  PSNR={bicubic_psnr:.2f} dB, SSIM={bicubic_ssim:.4f}")
print(f"LATINO:   PSNR={final_psnr:.2f} dB, SSIM={final_ssim:.4f}")
print(f"Improvement: +{final_psnr - bicubic_psnr:.2f} dB")
print("="*50)

## Visualize Results

In [ ]:
# Compare results
show_images(
    [gt_image, bicubic, result],
    ["Ground Truth", f"Bicubic ({bicubic_psnr:.2f} dB)", f"LATINO ({final_psnr:.2f} dB)"],
    figsize=(15, 5)
)

In [ ]:
# Show intermediate iterations
if intermediates:
    titles = [f"Iter {i+1}" for i in range(len(intermediates))]
    show_images(intermediates, titles, figsize=(4*len(intermediates), 4))

In [ ]:
# Save result
result_np = (result.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
Image.fromarray(result_np).save("latino_result.png")
print("Saved to latino_result.png")